#  CIS Department Voice Chatbot
**Pipeline:** Voice In → Whisper Transcription → Pinecone RAG → LangGraph LLM → TTS Voice Out


##  Section 1: Install Dependencies

In [ ]:
# Install all required packages
!pip install -q openai pinecone langgraph pydub
!apt-get install -q ffmpeg  # Required by pydub for MP3 conversion
print(" All dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 13.6 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
✅ All dependencies installed.


##  Section 2: API Keys & Configuration

In [ ]:
import os

#  Set API keys here


OPENAI_API_KEY  = "sk-proj---------------------------"
PINECONE_API_KEY = "pcsk_--------------------------------------"
PINECONE_INDEX_NAME = "cis-department-kb"
PINECONE_NAMESPACE  = "cis-department"

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("✅ API keys set.")

✅ API keys set.


##  Section 3: Voice Recording (Colab JS Widget)
Run the cell below. Click **Record**, speak your question, then click **Stop**.
The audio is automatically saved as `recorded_question.mp3`.

In [4]:
#  Colab JavaScript audio recorder
# Because Google Colab runs in a browser, sounddevice cannot access the mic.
# Instead, we use a JS widget that records via the browser's MediaRecorder API
# and sends the audio back to Python as base64-encoded bytes.
# Source/technique: standard Colab audio capture pattern used widely in ML notebooks.

from IPython.display import display, Javascript, Audio
from google.colab.output import eval_js
from base64 import b64decode
from pydub import AudioSegment
import io

RECORD_JS = """
const sleep = ms => new Promise(resolve => setTimeout(resolve, ms));

async function recordVoice() {
  const div = document.createElement('div');
  div.style.cssText = 'padding:10px;background:#1e1e2e;border-radius:8px;color:#cdd6f4;font-family:monospace;';
  document.querySelector('#output-area').appendChild(div);

  const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  const recorder = new MediaRecorder(stream);
  const chunks = [];

  recorder.ondataavailable = e => chunks.push(e.data);

  // Start button
  const startBtn = document.createElement('button');
  startBtn.textContent = '🎙️ Start Recording';
  startBtn.style.cssText = 'margin:5px;padding:8px 16px;background:#89b4fa;border:none;border-radius:4px;cursor:pointer;font-size:14px;';
  div.appendChild(startBtn);

  // Stop button
  const stopBtn = document.createElement('button');
  stopBtn.textContent = '⏹️ Stop Recording';
  stopBtn.disabled = true;
  stopBtn.style.cssText = 'margin:5px;padding:8px 16px;background:#f38ba8;border:none;border-radius:4px;cursor:pointer;font-size:14px;';
  div.appendChild(stopBtn);

  const status = document.createElement('p');
  status.textContent = 'Click Start to begin recording.';
  div.appendChild(status);

  startBtn.onclick = () => {
    recorder.start();
    startBtn.disabled = true;
    stopBtn.disabled = false;
    status.textContent = '🔴 Recording... Click Stop when done.';
  };

  await new Promise(resolve => {
    stopBtn.onclick = () => {
      recorder.stop();
      stream.getTracks().forEach(t => t.stop());
      status.textContent = '✅ Recording stopped. Processing...';
      resolve();
    };
  });

  await new Promise(resolve => recorder.onstop = resolve);

  const blob = new Blob(chunks, { type: 'audio/webm' });
  const reader = new FileReader();
  return new Promise(resolve => {
    reader.onloadend = () => resolve(reader.result.split(',')[1]);
    reader.readAsDataURL(blob);
  });
}

recordVoice();
"""

def record_voice(output_filename="recorded_question.mp3"):
    """
    Records audio from the browser microphone using a JS widget (Colab only).
    Converts the webm audio blob to MP3 via pydub and saves to disk.
    Returns the saved file path.
    """
    print("🎙️ Use the buttons below to record your question...")
    audio_b64 = eval_js(RECORD_JS)           # Runs JS, waits for base64 return
    audio_bytes = b64decode(audio_b64)        # Decode base64 → raw bytes (webm)

    # Convert webm → MP3 using pydub (requires ffmpeg)
    webm_buf = io.BytesIO(audio_bytes)
    sound = AudioSegment.from_file(webm_buf, format="webm")
    sound.export(output_filename, format="mp3")

    print(f"✅ Audio saved as '{output_filename}'")
    return output_filename

# ── Record now ──────────────────────────────────────────────────────────────
recorded_file = record_voice("recorded_question.mp3")

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


🎙️ Use the buttons below to record your question...
✅ Audio saved as 'recorded_question.mp3'


In [5]:
# ── Playback: listen to what was recorded ───────────────────────────────────
print("▶️ Playing back your recording:")
display(Audio(recorded_file, autoplay=True))

▶️ Playing back your recording:


## Section 4: Transcription with OpenAI Whisper

In [6]:
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)

def transcribe(audio_path: str) -> str:
    """
    Sends an MP3 file to OpenAI Whisper for speech-to-text transcription.
    Returns the transcribed text string.
    Source: OpenAI Whisper API docs + LangGraph Voice ChatBot class notebook.
    """
    with open(audio_path, "rb") as audio_file:
        response = client.audio.transcriptions.create(
            model="whisper-1",
            file=audio_file,
            response_format="text"
        )
    return response.strip()

#  Transcribe the recording
user_question = transcribe(recorded_file)
print(f"📝 Transcribed question:\n  '{user_question}'")

📝 Transcribed question:
  'What degree CIS department offer?'


##  Section 5: Pinecone Setup


In [7]:
from pinecone import Pinecone

# Connect to Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(PINECONE_INDEX_NAME)

print(f"✅ Connected to Pinecone index: '{PINECONE_INDEX_NAME}'")
print(index.describe_index_stats())

✅ Connected to Pinecone index: 'cis-department-kb'
DescribeIndexStatsResponse(dimension=1024, total_vector_count=20, metric='cosine', namespaces=1)


##  Section 6: Knowledge Base — CIS Department Text Chunks
All 18 chunks cover the three required sources:
- **Source 1:** CIS Dept Fact Sheet (degrees, curriculum, careers)
- **Source 2:** Student Outcomes (ABET outcomes 1–6)
- **Source 3:** UMass Dartmouth Student Handbook (policies, procedures)

In [8]:
#  Knowledge Base: 18 text chunks
# Each chunk is short and focused on one topic for best retrieval accuracy.

CIS_KNOWLEDGE_BASE = [

    #  Source 1: CIS Department Fact Sheet

    {
        "id": "cs-001",
        "text": (
            "UMass Dartmouth's Computer Science program offers a Bachelor of Science (BS) in Computer Science "
            "as well as a software engineering option. The program is housed in the College of Engineering. "
            "Students can also pursue minors in Computer Science, Game Design, or Mobile Applications."
        )
    },
    {
        "id": "cs-002",
        "text": (
            "The BS in Computer Science requires a total of 120 credits. Of those, 65 credits are in "
            "general engineering and computer science coursework. The software engineering option requires "
            "63 credits in the engineering and CS core instead of 65."
        )
    },
    {
        "id": "cs-003",
        "text": (
            "Core curriculum topics in the CIS program include algorithms, artificial intelligence, "
            "computer architecture, computer graphics, networks, databases, robotics, and software engineering. "
            "Students gain both theoretical foundations and practical skills across these areas."
        )
    },
    {
        "id": "cs-004",
        "text": (
            "Minor programs offered by the CIS department include: a Computer Science minor (21 credits), "
            "a Game Design minor (20 credits), and a Mobile Applications minor (17 credits). "
            "These minors are open to students from other majors."
        )
    },
    {
        "id": "cs-005",
        "text": (
            "All CS seniors complete a capstone project as part of the curriculum. "
            "The capstone is a team-based project that addresses real-world industry needs. "
            "It gives students hands-on experience working collaboratively on a substantial engineering problem."
        )
    },
    {
        "id": "cs-006",
        "text": (
            "CIS students have completed internships at companies such as Dell EMC, General Dynamics, "
            "Global Aquaculture Alliance, and the Naval Undersea Warfare Systems center. "
            "These placements provide valuable professional experience before graduation."
        )
    },
    {
        "id": "cs-007",
        "text": (
            "Graduates of the UMass Dartmouth Computer Science program have been hired by major employers "
            "including Amazon, Apple, Google, IBM, Lockheed Martin, MathWorks, Microsoft, and Oracle. "
            "The program has a strong track record of career placement across industry sectors."
        )
    },
    {
        "id": "cs-008",
        "text": (
            "Entry-level salaries for UMass Dartmouth CS graduates typically range from $58,990 to $74,784 "
            "per year. Salaries vary depending on the employer, location, and specific role."
        )
    },
    {
        "id": "cs-009",
        "text": (
            "The Computer Science program at UMass Dartmouth is accredited by the Computing Accreditation "
            "Commission (CAC) of ABET. ABET accreditation ensures the program meets rigorous quality "
            "standards recognized nationally and internationally by employers and graduate schools."
        )
    },
    {
        "id": "cs-010",
        "text": (
            "Students interested in graduate study can pursue an accelerated BS/MS degree that can be "
            "completed in 5 years. The department also offers a Master of Science (MS) in Computer Science "
            "and a PhD in Engineering and Applied Science for those seeking advanced degrees."
        )
    },

    #  Source 2: CIS Student Outcomes

    {
        "id": "so-001",
        "text": (
            "Student Outcome 1: Graduates can analyze complex computing problems and apply principles "
            "of computing and other relevant disciplines to identify solutions. "
            "This covers both theoretical analysis and practical problem-solving skills."
        )
    },
    {
        "id": "so-002",
        "text": (
            "Student Outcome 2: Graduates can design, implement, and evaluate a computing-based solution "
            "that meets a given set of computing requirements in their discipline."
        )
    },
    {
        "id": "so-003",
        "text": (
            "Student Outcome 3: Graduates can communicate effectively in a variety of professional contexts. "
            "This includes written documentation, oral presentations, and technical reports."
        )
    },
    {
        "id": "so-004",
        "text": (
            "Student Outcome 4: Graduates recognize professional responsibilities and make informed judgments "
            "in computing practice based on legal and ethical principles. "
            "Ethics and professional conduct are woven throughout the curriculum."
        )
    },
    {
        "id": "so-005",
        "text": (
            "Student Outcome 5: Graduates can function effectively as a member or leader of a team "
            "engaged in activities appropriate to the discipline. "
            "Teamwork skills are developed through group projects and the senior capstone."
        )
    },
    {
        "id": "so-006",
        "text": (
            "Student Outcome 6: Graduates apply computer science theory and software development "
            "fundamentals to produce computing-based solutions. "
            "This combines algorithmic thinking with practical software engineering skills."
        )
    },

    #  Source 3: UMass Dartmouth Student Handbook

    {
        "id": "hb-001",
        "text": (
            "The UMass Dartmouth Student Affairs office is located on the 2nd floor of the Campus Center. "
            "Students can contact the office by phone at 508-999-8995. "
            "The office handles a wide range of student support services and policy questions."
        )
    },
    {
        "id": "hb-002",
        "text": (
            "The UMass Dartmouth Student Handbook outlines academic regulations and procedures that all "
            "students must follow. This includes policies on attendance, academic integrity, grade appeals, "
            "and adding or dropping courses within the registration deadlines."
        )
    },
    {
        "id": "hb-003",
        "text": (
            "The Student Handbook describes the university's student conduct policies and community standards. "
            "Students are expected to uphold academic honesty and treat all members of the university "
            "community with respect. Violations may result in disciplinary action."
        )
    },
    {
        "id": "hb-004",
        "text": (
            "In the event of severe weather or emergencies, UMass Dartmouth follows snow and emergency "
            "closing procedures. Students should monitor the university's website, email alerts, and "
            "local news for announcements about campus closures or delayed openings."
        )
    },
]

print(f"✅ Knowledge base loaded: {len(CIS_KNOWLEDGE_BASE)} chunks ready to upload.")

✅ Knowledge base loaded: 20 chunks ready to upload.


## Section 7: Embed & Upload Chunks to Pinecone
 **Run this section only once.** After the chunks are in Pinecone,don't need to re-upload them.

In [9]:
import time

def embed_text(text: str) -> list[float]:
    """
    Converts a text string into a 1024-dimension vector using
    OpenAI's text-embedding-3-small model.
    """
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text,
        dimensions=1024  #  match Pinecone index dimension
    )
    return response.data[0].embedding


def upload_knowledge_base(chunks: list[dict], namespace: str, batch_size: int = 10):
    """
    Embeds each chunk and upserts it into Pinecone.
    Uses batching to stay within Pinecone's rate limits.
    """
    vectors = []
    for i, chunk in enumerate(chunks):
        print(f"  Embedding chunk {i+1}/{len(chunks)}: {chunk['id']}")
        vector = embed_text(chunk["text"])
        vectors.append({
            "id": chunk["id"],
            "values": vector,
            "metadata": {"text": chunk["text"]}
        })

        # Upload in batches
        if len(vectors) == batch_size or i == len(chunks) - 1:
            index.upsert(vectors=vectors, namespace=namespace)
            print(f"  ✅ Upserted batch of {len(vectors)} chunks.")
            vectors = []
            time.sleep(1)  # Brief pause to respect rate limits

    print(f"\n🎉 All {len(chunks)} chunks uploaded to namespace '{namespace}'.")


#  Upload now
print(" Uploading CIS department knowledge base to Pinecone...")
#upload_knowledge_base(CIS_KNOWLEDGE_BASE, namespace=PINECONE_NAMESPACE)

 Uploading CIS department knowledge base to Pinecone...


##  Section 8: RAG Retrieval Function

In [10]:
def retrieve_context(question: str, top_k: int = 3) -> str:
    """
    Embeds the question, queries Pinecone for the top_k most relevant
    chunks, and returns them as a single joined string.
    Source: class notebook RAG pattern.
    """
    question_vector = embed_text(question)
    results = index.query(
        vector=question_vector,
        top_k=top_k,
        include_metadata=True,
        namespace=PINECONE_NAMESPACE
    )
    # Extract text from each matched chunk
    context_pieces = [match["metadata"]["text"] for match in results["matches"]]
    return "\n\n".join(context_pieces)


#  Quick retrieval test
test_context = retrieve_context("What degrees does the CIS department offer?")
print("🔍 Test retrieval result:")
print(test_context)

🔍 Test retrieval result:
The BS in Computer Science requires a total of 120 credits. Of those, 65 credits are in general engineering and computer science coursework. The software engineering option requires 63 credits in the engineering and CS core instead of 65.

The UMass Dartmouth Student Handbook outlines academic regulations and procedures that all students must follow. This includes policies on attendance, academic integrity, grade appeals, and adding or dropping courses within the registration deadlines.

In the event of severe weather or emergencies, UMass Dartmouth follows snow and emergency closing procedures. Students should monitor the university's website, email alerts, and local news for announcements about campus closures or delayed openings.


## 🤖 Section 9: LangGraph RAG Chatbot

In [11]:
from langgraph.graph import StateGraph, END
from langgraph.graph.message import MessagesState
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

#  System prompt: CIS department assistant
RAG_PROMPT = """\
You are a helpful voice assistant for the Computer and Information Science (CIS) department
at UMass Dartmouth. Answer the student's question using ONLY the context provided below.
If the context does not contain enough information to answer fully, say so honestly
and suggest the student contact the department directly.

Keep your answers clear, friendly, and concise — they will be read aloud as speech.
Avoid bullet points or markdown formatting; use plain spoken sentences.

CONTEXT:
{context}
"""


def rag_chatbot(state: MessagesState) -> dict:
    """
    LangGraph node: retrieves relevant context from Pinecone, then calls
    the OpenAI LLM to generate a grounded answer.
    Source: class LangGraph Voice ChatBot notebook, adapted for CIS dept.
    """
    # Get the latest user message
    user_message = state["messages"][-1].content

    # Retrieve relevant context from Pinecone
    context = retrieve_context(user_message, top_k=3)

    # Build the system prompt with injected context
    system_prompt = RAG_PROMPT.format(context=context)

    # Build the message list for the LLM
    messages_for_llm = [
        {"role": "system", "content": system_prompt},
    ]
    # Include full conversation history for multi-turn memory
    for msg in state["messages"]:
        if isinstance(msg, HumanMessage):
            messages_for_llm.append({"role": "user", "content": msg.content})
        elif isinstance(msg, AIMessage):
            messages_for_llm.append({"role": "assistant", "content": msg.content})

    # Call the LLM
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages_for_llm,
    )
    answer = response.choices[0].message.content

    return {"messages": state["messages"] + [AIMessage(content=answer)]}


# Build the LangGraph StateGraph
memory = MemorySaver()   # Enables multi-turn conversation memory

graph_builder = StateGraph(MessagesState)
graph_builder.add_node("rag_chatbot", rag_chatbot)
graph_builder.set_entry_point("rag_chatbot")
graph_builder.add_edge("rag_chatbot", END)

chatbot_graph = graph_builder.compile(checkpointer=memory)

print(" LangGraph RAG chatbot compiled and ready.")

 LangGraph RAG chatbot compiled and ready.


##  Section 10: Text-to-Speech with OpenAI TTS

In [12]:

def speak(text: str, voice: str = "nova", output_path: str = "response.mp3") -> str:
    """
    Converts text to speech using OpenAI TTS and saves the result as an MP3.
    Available voices: alloy, echo, fable, onyx, nova, shimmer.
    Source: OpenAI TTS API docs + class notebook.
    """
    response = client.audio.speech.create(
        model="tts-1",
        voice=voice,
        input=text
    )
    response.stream_to_file(output_path)
    print(f"✅ TTS audio saved to '{output_path}'")
    return output_path


#  Quick TTS test
test_audio = speak("Hello! I am the CIS department assistant. How can I help you today?")
display(Audio(test_audio, autoplay=True))

✅ TTS audio saved to 'response.mp3'


/tmp/ipykernel_811/1478082836.py:12: DeprecationWarning: Due to a bug, this method doesn't actually stream the response content, `.with_streaming_response.method()` should be used instead
  response.stream_to_file(output_path)


## Section 11: Full End-to-End Pipeline
This is the main function that ties everything together:
**Record → Transcribe → Retrieve → LLM Answer → Speak → Play**

In [13]:
# Conversation thread ID (fixed so MemorySaver tracks history across turns)
THREAD_ID = {"configurable": {"thread_id": "cis-session-1"}}


def run_voice_chatbot():
    """
    Full pipeline:
      1. Record voice from browser microphone (Colab JS widget)
      2. Play back the recording so user can verify
      3. Transcribe with Whisper
      4. Run LangGraph RAG chatbot (retrieves from Pinecone + calls LLM)
      5. Convert answer to speech with TTS
      6. Play the spoken answer inline
    """
    print("=" * 60)
    print("🎙️  CIS Department Voice Assistant")
    print("=" * 60)

    #  Step 1: Record
    audio_file = record_voice("recorded_question.mp3")

    #  Step 2: Playback of recording
    print("\n▶️  Your recording:")
    display(Audio(audio_file, autoplay=False))   # autoplay=False so user controls

    # Step 3: Transcribe
    print("\n📝 Transcribing...")
    question = transcribe(audio_file)
    print(f"   You asked: '{question}'")

    # Step 4: LangGraph RAG answer
    print("\n Generating answer...")
    result = chatbot_graph.invoke(
        {"messages": [HumanMessage(content=question)]},
        config=THREAD_ID
    )
    answer = result["messages"][-1].content
    print(f"   Answer: {answer}")

    #  Step 5: Text-to-Speech
    print("\n🔊 Converting answer to speech...")
    response_audio = speak(answer, voice="nova", output_path="response.mp3")

    #  Step 6: Play the spoken answer
    print("\n▶️  CIS Assistant reply:")
    display(Audio(response_audio, autoplay=True))

    print("\n" + "=" * 60)
    print("✅ Done! Run this cell again to ask another question.")
    print("=" * 60)


#  Run the chatbot
run_voice_chatbot()

🎙️  CIS Department Voice Assistant
🎙️ Use the buttons below to record your question...
✅ Audio saved as 'recorded_question.mp3'

▶️  Your recording:



📝 Transcribing...
   You asked: 'this artificial intelligence is available for undergrad program'

 Generating answer...
   Answer: The information provided does not specify if there is a specific undergraduate program focused on artificial intelligence at UMass Dartmouth. I recommend contacting the Computer and Information Science department directly for more details on this topic.

🔊 Converting answer to speech...
✅ TTS audio saved to 'response.mp3'

▶️  CIS Assistant reply:


/tmp/ipykernel_811/1478082836.py:12: DeprecationWarning: Due to a bug, this method doesn't actually stream the response content, `.with_streaming_response.method()` should be used instead
  response.stream_to_file(output_path)



✅ Done! Run this cell again to ask another question.


**Project-2**

**Server URLs**

In [ ]:
MCP_SERVER_URL = "https://mcp-server-production-6150.up.railway.app"
A2A_SERVER_URL = "https://a2a-server-production-f1a0.up.railway.app"
print("Server URLs set.")

✅ Server URLs set.


**MCP Test**


In [15]:
import requests

r = requests.get(MCP_SERVER_URL + "/tools")
print("MCP Tools:", r.json())

MCP Tools: [{'description': 'Return the current date and time as a string.', 'name': 'get_current_time'}, {'description': 'Compute GPA from a list of grades.\nEach item looks like {"grade": "A", "credits": 3}.\nLetter grades supported: A, A-, B+, B, B-, C+, C, C-, D+, D, F.', 'name': 'gpa_calculator'}, {'description': 'Convert a temperature between Fahrenheit and Celsius.\nfrom_unit must be "F" or "C".', 'name': 'convert_temperature'}, {'description': 'Look up a CIS course by code, e.g. "CIS 360".\nReturns the course name, credit hours, and instructor.', 'name': 'lookup_course'}, {'description': 'Look up the definition of a CS or AI term from a small built-in glossary.', 'name': 'define_term'}]


**A2A Test**

In [16]:
# Quiz
r = requests.post(A2A_SERVER_URL + "/quiz/execute/quiz", json={"text": "binary search trees"})
print("QUIZ:", r.json())

# Summarizer
r = requests.post(A2A_SERVER_URL + "/summarizer/execute/summarize", json={"text": "Artificial intelligence is a branch of computer science."})
print("SUMMARIZE:", r.json())

# Translator
r = requests.post(A2A_SERVER_URL + "/translator/execute/translate", json={"text": "Hello, how are you?"})
print("TRANSLATE:", r.json())

QUIZ: {'result': 'Topic: binary search trees\nQ: In a binary search tree, where does a value smaller than the root go?\nA) Right subtree\nB) Left subtree\nC) At the root\nD) Anywhere\nAnswer: B', 'skill': 'quiz', 'status': 'success'}
SUMMARIZE: {'result': 'Artificial intelligence is a branch of computer science.', 'skill': 'summarize', 'status': 'success'}
TRANSLATE: {'result': '¿Hola, cómo estás?', 'skill': 'translate', 'status': 'success'}


**MCP All Tools Test**

In [18]:
# Time
r = requests.post(MCP_SERVER_URL + "/call", json={"tool_name": "get_current_time", "arguments": {}})
print("TIME:", r.json())

# Temperature
r = requests.post(MCP_SERVER_URL + "/call", json={"tool_name": "convert_temperature", "arguments": {"value": 75, "from_unit": "F"}})
print("TEMP:", r.json())

# Course
r = requests.post(MCP_SERVER_URL + "/call", json={"tool_name": "lookup_course", "arguments": {"course_code": "CIS 360"}})
print("COURSE:", r.json())

TIME: {'result': '2026-05-12 01:23:47', 'status': 'success'}
TEMP: {'result': '75 F = 23.9 C', 'status': 'success'}
COURSE: {'result': 'CIS 360: Data Structures and Algorithms (3 credits, taught by Dr. Long)', 'status': 'success'}


**PROJECT 2 — MCP + A2A + RAG Integration**

In [26]:

import re

#  Task 4: MCP Cache
mcp_cache = {}

#  MCP Helper
def call_mcp(tool_name, arguments={}):
    cache_key = f"{tool_name}_{str(arguments)}"
    if cache_key in mcp_cache:
        print(f"[memory hit] '{tool_name}' served from cache")
        return mcp_cache[cache_key]
    r = requests.post(MCP_SERVER_URL + "/call",
                      json={"tool_name": tool_name, "arguments": arguments})
    result = r.json().get("result", "No result")
    mcp_cache[cache_key] = result
    print(f"[MCP] {tool_name} → {result}")
    return result

#  A2A Helper
def call_a2a(agent, skill, text):
    r = requests.post(f"{A2A_SERVER_URL}/{agent}/execute/{skill}",
                      json={"text": text}, timeout=30)
    result = r.json().get("result", "No result")
    print(f"[A2A] {agent}/{skill} → {result}")
    return result

#  Router
def route(user_input):
    text = user_input.lower()

    if any(w in text for w in ["quiz", "test me", "quiz me"]):
        topic = re.sub(r"quiz me on|quiz me about|quiz me|test me on", "", text).strip()
        return call_a2a("quiz", "quiz", topic)

    elif any(w in text for w in ["summarize", "shorten", "summary"]):
        return call_a2a("summarizer", "summarize", user_input)

    elif any(w in text for w in ["translate", "spanish", "in spanish"]):
        return call_a2a("translator", "translate", user_input)

    elif any(w in text for w in ["time", "date", "what time"]):
        return call_mcp("get_current_time")

    elif any(w in text for w in ["fahrenheit", "celsius", "convert", "temperature"]):
        nums = re.findall(r'\d+\.?\d*', user_input)
        value = float(nums[0]) if nums else 0
        unit = "F" if "fahrenheit" in text or "f to c" in text else "C"
        return call_mcp("convert_temperature", {"value": value, "from_unit": unit})

    elif any(w in text for w in ["gpa", "grade", "grades"]):
        grades = [{"grade": "A", "credits": 3}, {"grade": "B", "credits": 3}]
        return call_mcp("gpa_calculator", {"grades": grades})

    elif any(w in text for w in ["cis", "course", "class"]):
        match = re.search(r'CIS\s?\d+', user_input, re.IGNORECASE)
        course_code = match.group(0).upper() if match else "CIS 360"
        return call_mcp("lookup_course", {"course_code": course_code})

    elif any(w in text for w in ["define", "what is", "meaning"]):
        term = re.sub(r"define|what is|meaning of", "", text).strip()
        return call_mcp("define_term", {"term": term})

    else:
        print("[RAG] Using Pinecone retrieval...")
        result = chatbot_graph.invoke(
            {"messages": [HumanMessage(content=user_input)]},
            config=THREAD_ID
        )
        return result["messages"][-1].content

#  Full Voice Pipeline
def run_voice_chatbot_v2():
    print("=" * 55)
    print("🎙️  CIS Voice Assistant v2 — MCP + A2A + RAG")
    print("=" * 55)

    # Step 1: Record
    audio_file = record_voice("recorded_question.mp3")
    display(Audio(audio_file, autoplay=False))

    # Step 2: Transcribe
    question = transcribe(audio_file)
    print(f"\n📝 You asked: '{question}'")

    # Step 3: Route
    print("\n🔀 Routing...")
    answer = route(question)
    print(f"\n🤖 Answer: {answer}")

    # Step 4: Speak
    response_audio = speak(answer, voice="nova")
    display(Audio(response_audio, autoplay=True))
    print("\n✅ Done! Run again for another question.")

#  Run!
run_voice_chatbot_v2()

🎙️  CIS Voice Assistant v2 — MCP + A2A + RAG
🎙️ Use the buttons below to record your question...
✅ Audio saved as 'recorded_question.mp3'



📝 You asked: 'What time is it?'

🔀 Routing...
[MCP] get_current_time → 2026-05-12 01:31:23

🤖 Answer: 2026-05-12 01:31:23
✅ TTS audio saved to 'response.mp3'


/tmp/ipykernel_811/1478082836.py:12: DeprecationWarning: Due to a bug, this method doesn't actually stream the response content, `.with_streaming_response.method()` should be used instead
  response.stream_to_file(output_path)



✅ Done! Run again for another question.


In [27]:
# Check: Wrong tool name should not crash
result = call_mcp("wrong_tool_name", {})
print("Result:", result)

[MCP] wrong_tool_name → No result
Result: No result


In [28]:
# Cache test — second call should show [memory hit]
print("First call:")
result1 = call_mcp("get_current_time", {})

print("\nSecond call (should be [memory hit]):")
result2 = call_mcp("get_current_time", {})

First call:
[memory hit] 'get_current_time' served from cache

Second call (should be [memory hit]):
[memory hit] 'get_current_time' served from cache


In [29]:

# TASK 4 — Memory Cache Demo
# Reset cache to demonstrate fresh caching behavior
mcp_cache = {}
print(" Cache cleared. Starting fresh demo...\n")

print("=" * 50)
print("TASK 4: MCP Memory Cache Demonstration")
print("=" * 50)

print("\n  First call — fetches from MCP server:")
result1 = call_mcp("get_current_time", {})

print("\n  Second call — same tool, same arguments:")
result2 = call_mcp("get_current_time", {})

print("\n" + "=" * 50)
print("Cache working correctly!")
print(f"   Both results match: {result1 == result2}")
print("=" * 50)

 Cache cleared. Starting fresh demo...

TASK 4: MCP Memory Cache Demonstration

  First call — fetches from MCP server:
[MCP] get_current_time → 2026-05-12 01:39:38

  Second call — same tool, same arguments:
[memory hit] 'get_current_time' served from cache

Cache working correctly!
   Both results match: True
